# Simulación reproducible de manos independientes de 5 cartas

Este notebook usa el evaluador validado en `src/poker_sim/`. Cada mano se extrae de un mazo completo y es independiente de las demás.

Ejecutá todas las celdas desde el principio (`Run All`).

In [ ]:
from pathlib import Path
from random import Random
import sys

RAIZ_PROYECTO = Path.cwd()
sys.path.insert(0, str(RAIZ_PROYECTO / "src"))

from poker_sim import evaluar_mano, mazo_estandar, resumir_estadisticas

SEMILLA = 20260805
NUMERO_DE_MANOS = 100_000

## Generación

`Random` es local al experimento. Al reiniciarlo con la misma semilla se obtiene exactamente la misma muestra.

In [ ]:
mazo = mazo_estandar()
generador = Random(SEMILLA)

def generar_mano():
    """Genera una mano independiente de cinco cartas sin repetición."""
    return tuple(generador.sample(mazo, k=5))

manos = [generar_mano() for _ in range(NUMERO_DE_MANOS)]

## Comparación estadística

Cada fila muestra el resultado de la simulación junto al valor exacto para todas las manos de 5 cartas. **Esperadas** es la cantidad teórica media para este tamaño de muestra; no tiene por qué ser un número entero. El **error** es `observado − teórico`, en puntos porcentuales (pp).

In [ ]:
evaluaciones = [evaluar_mano(mano) for mano in manos]
filas = resumir_estadisticas(evaluaciones)

print(f"Semilla: {SEMILLA} | Manos simuladas: {NUMERO_DE_MANOS:,}")
print(f"Ejemplo: {' '.join(map(str, manos[0]))} ({evaluaciones[0].nombre})\n")
print(f"{'Categoría':22} {'Obs.':>8} {'Observado':>11} {'Teórico':>11} {'Esperadas':>11} {'Error (pp)':>12}")
print("-" * 84)

for fila in filas:
    print(
        f"{fila.categoria.name:22} "
        f"{fila.observadas:>8,} "
        f"{fila.porcentaje_observado:>10.4%} "
        f"{fila.porcentaje_teorico:>10.4%} "
        f"{fila.esperadas_en_muestra:>11.2f} "
        f"{fila.error_puntos_porcentuales:>+11.4f}"
    )

## Cómo interpretar el resultado

Las diferencias no prueban que el simulador esté mal: son el error muestral esperado de una simulación aleatoria finita. Al aumentar `NUMERO_DE_MANOS`, los porcentajes observados suelen acercarse a los teóricos; al cambiar `SEMILLA`, cambia la muestra y también cambian las diferencias.

## Comprobaciones

Las comprobaciones confirman que la simulación respeta el mazo y que se evaluaron todas las manos. Las pruebas exhaustivas de la matemática viven en `tests/test_evaluator.py`.

In [ ]:
assert len(mazo) == 52
assert all(len(mano) == 5 and len(set(mano)) == 5 for mano in manos)
assert sum(fila.observadas for fila in filas) == NUMERO_DE_MANOS

print("Simulación válida.")